# PasteTrace — Mamba Behavioral Sequence Model

**Yêu cầu:** Vào `Runtime → Change runtime type → T4 GPU` trước khi chạy.

Dataset: **205 sinh viên tổng hợp** trong `test_new_cohort/` (đã có sẵn trong repo — không cần upload Drive).

## Pipeline
```
Bước 0  Setup   (kiểm tra GPU, cài thư viện, clone repo)
Bước 1  Build   (meta.json → chuỗi sự kiện, data/train_sequences/)
Bước 2  Train   (Mamba model, lưu models/mamba/mamba.pt)
Bước 3  Lưu     (download mamba.pt về máy hoặc lên Drive)
Bước 4  Predict (chạy inference trên thư mục tests/)
```

## Bước 0a — Kiểm tra GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️ GPU not enabled. Enable in Notebook Settings → GPU')

## Bước 0b — Cài thư viện Mamba (~3 phút lần đầu)

In [ ]:
import os
import subprocess
import importlib.util
import torch

print("="*60)
print("Torch :", torch.__version__)
print("CUDA  :", torch.version.cuda)
print("="*60)

if importlib.util.find_spec("mamba_ssm") is None:

    print("\nInstalling build dependencies...")
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
        capture_output=True
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", 
         "build-essential", "cuda-toolkit"],
        check=True,
        capture_output=True
    )

    print("Installing Python build tools...")
    subprocess.run(
        [
            "pip","install","-q","--upgrade",
            "pip", "setuptools", "wheel", "ninja",
            "packaging", "pybind11", "cmake"
        ],
        check=True
    )

    print("\nInstalling causal-conv1d...")
    result = subprocess.run(
        ["pip","install","-q","causal-conv1d>=1.1.0"],
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        print("⚠️  causal-conv1d build failed, trying pre-built wheel...")
        subprocess.run(
            ["pip","install","-q","--no-build-isolation","causal-conv1d"],
            check=True
        )

    print("\nInstalling mamba-ssm...")
    result = subprocess.run(
        ["pip","install","-q","mamba-ssm>=0.1"],
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        print("⚠️  mamba-ssm build failed, trying pre-built wheel...")
        subprocess.run(
            ["pip","install","-q","--no-build-isolation","mamba-ssm"],
            check=True
        )

print("\nInstalling other packages...")
subprocess.run(
    [
        "pip","install","-q",
        "pandas",
        "scikit-learn",
        "plotly"
    ],
    check=True
)

# Kiểm tra import
try:
    from mamba_ssm import Mamba
    print("\n✓ SUCCESS! All dependencies installed")
except ImportError as e:
    print(f"\n✗ ERROR: {e}")
    raise

## Bước 0c — Clone repo từ GitHub

Lần đầu: clone repo về. Lần sau (runtime mới): clone lại hoặc pull update.

In [ ]:
import sys

REPO_URL = 'https://github.com/lequocviet-3103/Fraud-Detection.git'
REPO_DIR = '/kaggle/working/Fraud-Detection'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo da co, pull update...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())
!ls

## Bước 0d — Kiểm tra data

Dataset `test_new_cohort/` đã có sẵn trong repo (205 sinh viên tổng hợp — không cần upload Drive).

In [ ]:
DATA_DIR = 'test_new_cohort'
if os.path.isdir(DATA_DIR):
    cases = sorted([d for d in os.listdir(DATA_DIR)
                    if os.path.isdir(os.path.join(DATA_DIR, d))])
    total = 0
    for c in cases:
        students = [s for s in os.listdir(os.path.join(DATA_DIR, c))
                    if os.path.isdir(os.path.join(DATA_DIR, c, s))]
        print(f'  {c}: {len(students)} students')
        total += len(students)
    print(f'\nOK — {total} students total in {len(cases)} cases')
else:
    print('CANH BAO: Khong tim thay test_new_cohort/. Chay lai Buoc 0c (git pull).')

## Bước 1 — Build Sequences

Đọc `meta.json` → vector sự kiện T/P/C → `data/train_sequences/`

Dùng flag `--data-dir test_new_cohort` để trỏ vào dataset tổng hợp.

In [ ]:
!python -m src.data.build_sequences --data-dir test_new_cohort --min-events 3

import pandas as pd
df = pd.read_csv('data/sequences_index.csv')
print(df[['id','label','n_events','time_available']].to_string())
print(f'\nTong: {len(df)} sinh vien | cheat={sum(df.label==1)} | normal={sum(df.label==0)}')
print(f'Events: min={df.n_events.min()}  median={df.n_events.median():.0f}  max={df.n_events.max()}')

## Bước 2 — Train

Train model trên toàn bộ dataset.

In [ ]:
!python -m src.models.mamba_model train \
    --d-model 64 --n-layers 2 --dropout 0.2 \
    --epochs 80 --lr 1e-3 --patience 10 \
    --batch-size 8 --max-len 1000

print("\n" + "="*60)
print("Checking model files after training...")
print("="*60)

# Kiem tra model files
import os
for fname in ['mamba.pt', 'config.json', 'scaler.json']:
    p = f'models/mamba/{fname}'
    if os.path.isfile(p):
        size_kb = os.path.getsize(p) / 1024
        print(f'✓ OK     {p}  ({size_kb:.1f} KB)')
    else:
        print(f'✗ MISSING  {p}')

## Bước 3 — Lưu model

In [ ]:

import zipfile

with zipfile.ZipFile('/kaggle/working/mamba_trained.zip', 'w') as z:
    for f in ['models/mamba/mamba.pt', 'models/mamba/config.json',
              'models/mamba/scaler.json']:
        if os.path.isfile(f):
            z.write(f)
            print(f'Added {f}')

print('✓ Saved to /kaggle/working/mamba_trained.zip')
print('File co the download tu Kaggle Output tab')

import shutil

SAVE_DIR = '/kaggle/working/trained_models'
os.makedirs(SAVE_DIR, exist_ok=True)

for src in ['models/mamba/mamba.pt', 'models/mamba/config.json',
            'models/mamba/scaler.json']:
    if os.path.isfile(src):
        shutil.copy2(src, SAVE_DIR)
        print(f'Saved {os.path.basename(src)} -> /kaggle/working')

print(f'\n✓ Model saved in: {SAVE_DIR}')
print('Download từ Kaggle Output tab')


---
## Lần sau — Load model từ Drive (bỏ qua bước Train)

Khi mở Colab mới, chạy lại Bước 0a → 0b → 0c, rồi chạy cell này:

In [ ]:
import shutil

# Nếu bạn upload output thành Kaggle Dataset, thay "your-dataset-name"
KAGGLE_MODEL = '/kaggle/input/your-dataset-name/trained_models'

os.makedirs('models/mamba', exist_ok=True)

for fname in ['mamba.pt', 'config.json', 'scaler.json']:
    try:
        shutil.copy2(f'{KAGGLE_MODEL}/{fname}', f'models/mamba/{fname}')
        print(f'✓ Loaded {fname}')
    except FileNotFoundError:
        print(f'✗ Not found: {fname}')

# Predict sinh vien moi
STUDENT_FOLDER = 'test_new_cohort/212/S01'
!python -m src.models.mamba_model predict {STUDENT_FOLDER}

---
## Bước 4 — Predict trên tập tests/

Chạy model để đưa ra dự đoán (Normal/Cheat) trên bộ dữ liệu `tests/`.

In [ ]:
!python predict_tests.py tests

---
## Lỗi thường gặp

| Lỗi | Fix |
|-----|-----|
| `RuntimeError: GPU chua bat` | Runtime → Change runtime type → T4 GPU |
| `Getting requirements to build wheel` | Dùng `--no-build-isolation` (Bước 0b đã sửa) |
| `mamba_ssm not found` | Chạy lại Bước 0b |
| `sequences_index.csv not found` | Chạy Bước 1 |
| `test_new_cohort/ not found` | Chạy lại Bước 0c (git pull) |
| Session bị reset sau 12h | Chạy lại Bước 0a → 0c, load model từ Drive (cell cuối) |